## For beta GMs, construct BIDS compatible event files.ttsv

Niklas:"
task timings sind unter .onsets, die 40x6 doubles für trials x blocks"

block = run

BIDS requires:
- [onset    duration    trial_type] - could be more columns
- Onsets must be relative to the start of that run
- 

asd_task_data.mat:
- 94 subjecst x 6 (9?) runs x 40 trials
- entries: 'script_start', 'connection_time', 'block_loop_start', 'block', 'fixation', 'choice', 'response', 'feedback', 'block_loop_end'

### recode SUBID to BIDSID
subj_mapping = readtable(fullfile(folders.prepro,'participants.xlsx'));
subj_mapping.SUBID = cellfun(@(x) str2double(x(:,end-3:end)), subj_mapping.data_id);
targetID = subj_mapping.SUBID(strcmp(subj,subj_mapping.participant_id));

In [ ]:
import pandas as pd
import numpy as np
from scipy.io import loadmat
import os.path as op
import os 

fn_all_onsets = '/Users/mrenke/Desktop/asd_autism_project/data/asds_task_data.mat'

data_all_onsets = loadmat(fn_all_onsets, struct_as_record=False, squeeze_me=True)

bids_folder = '/Users/mrenke/data/ds-asd'
event_types = ['fixation', 'choice', 'response', 'feedback'] # subset from onsets.keys()

In [ ]:
fn_subID_bidsID_table = '/Users/mrenke/Desktop/asd_autism_project/participants.tsv'
subID_bidsID_table = pd.read_csv(fn_subID_bidsID_table, sep='\t')

# File that Niklas gave me in Slack seems wrong, participants file from Gökis drive seems correct
#fn_subID_bidsID_table = '/Users/mrenke/Desktop/asd_autism_project/data/participants.xlsx'
#subID_bidsID_table = pd.read_excel(fn_subID_bidsID_table)

bids_folder = '/Users/mrenke/data/ds-asd2'


In [163]:
# get BIDS ID lookup table
df_map = subID_bidsID_table.copy()
df_map["mat_id"] = (df_map["data_id"].str.extract(r"S(\d+)").astype(int))
df_map['BIDS_ID'] = df_map['participant_id'].str.extract(r'sub-(\d+)').astype(int)

id_lookup = dict(zip(df_map["mat_id"], df_map["BIDS_ID"]))

In [164]:
# function to handle nested mat objects
def todict(matobj):
    data = {}
    for field in matobj._fieldnames:
        elem = getattr(matobj, field)
        if isinstance(elem, type(matobj)):
            data[field] = todict(elem)
        else:
            data[field] = elem
    return data


In [165]:
for s in range(len(data_all_onsets['data'])):
    try:
        subj = todict(data_all_onsets['data'][s])
        subj_ID = subj['subjID']

        subj_BIDS_ID = id_lookup[subj_ID]
        target_folder = op.join(bids_folder, f'sub-{subj_BIDS_ID:02d}', 'ses-1', 'func')
        os.makedirs(target_folder, exist_ok=True)

        onsets = subj['onsets']
        block_starts = onsets['block']
        N_runs = len(block_starts)

        for run in range(N_runs):
            
            rows = []
            for ev in event_types:
                
                ev_matrix = onsets[ev]      # 40 x 6
                ev_run_onsets = ev_matrix[:, run] # - block_starts[run] --> seems like absolute experiment timestamps already!
                
                i = 1
                for onset in ev_run_onsets:
                    rows.append({
                        "trial_nr": i,
                        "onset": onset,
                        "duration": 0.0,      # for GLMsingle only the events counts
                        "trial_type": ev
                    })
                    i += 1
            
            df = pd.DataFrame(rows)
            df = df.sort_values("onset")
            
            fname = f"sub-{subj_BIDS_ID:02d}_ses-1_task-chase_run-{run+1}_events.tsv"
            df.to_csv(op.join(target_folder, fname), sep="\t", index=False)
    except Exception as e:
        print(f"Error processing subject {s} (mat ID: {subj_ID}): {e}")

## Explore data structure

In [98]:
data_all_onsets['data'][0].__dict__.keys()

dict_keys(['_fieldnames', 'subjID', 'taskID', 'fmriID', 'group', 'n_trials', 'n_blocks', 'bot_level', 'choice_own', 'choice_other', 'score_own', 'score_other', 'missing', 'trial', 'RT', 'onsets', 'date', 'bonus'])

In [99]:
data_all_onsets['data'][0].subjID

2001

In [90]:
subj = data_all_onsets['data'][0]   # first subject
onsets = subj.onsets


In [91]:
subj.onsets.__dict__.keys()


dict_keys(['_fieldnames', 'script_start', 'connection_time', 'block_loop_start', 'block', 'fixation', 'choice', 'response', 'feedback', 'block_loop_end'])

In [93]:
np.shape(subj.onsets.choice)

(40, 6)

In [67]:
onsets = todict(subj.onsets)

In [68]:
onsets.keys()

dict_keys(['script_start', 'connection_time', 'block_loop_start', 'block', 'fixation', 'choice', 'response', 'feedback', 'block_loop_end'])

In [69]:
onsets['block'].shape

(6,)

In [70]:
event_types = ['fixation', 'choice', 'response', 'feedback']

In [130]:
s = 0   # first subject

data_all_onsets['data'][0].subjID

subj = todict(data_all_onsets['data'][s])
subj_ID = subj['subjID']

subj_BIDS_ID = id_lookup[subj_ID][7:]
target_folder = op.join(bids_folder, f'sub-{subj_BIDS_ID}', 'ses-1', 'func')
os.makedirs(target_folder, exist_ok=True)

onsets = subj['onsets']
block_starts = onsets['block']
N_runs = len(block_starts)

for run in range(N_runs):
    
    rows = []
    for ev in event_types:
        
        ev_matrix = onsets[ev]      # 40 x 6
        ev_run_onsets = ev_matrix[:, run] # - block_starts[run] --> seems like absolute experiment timestamps already!
        
        i = 1
        for onset in ev_run_onsets:
            rows.append({
                "trial_nr": i,
                "onset": onset,
                "duration": 0.0,      # update if needed
                "trial_type": ev
            })
            i += 1
    
    df = pd.DataFrame(rows)
    df = df.sort_values("onset")
    
    fname = f"sub-{subj_BIDS_ID}_task-chase_run-{run+1}_events.tsv"
    df.to_csv(op.join(target_folder, fname), sep="\t", index=False)

In [127]:
fname

'sub-05_task-chase_run-6_events.tsv'

In [119]:
id_lookup = dict(zip(df_map["mat_id"], df_map["participant_id"]))


In [122]:
id_lookup[1001][7:]

'02'

In [86]:
df.set_index("trial_nr")

,onset,duration,trial_type
trial_nr,,,
1,0.017341,0.0,fixation
1,2.402906,0.0,choice
1,2.853465,0.0,response
1,8.658681,0.0,feedback
2,10.677234,0.0,fixation
...,...,...,...
39,426.295417,0.0,feedback
40,428.313961,0.0,fixation
40,432.718032,0.0,choice


In [82]:
block_starts

array([4597.0583437, 5162.1983295, 5831.2345458, 6346.3783618,
       7159.5977962, 7723.4867201])

In [84]:
ev_matrix

array([[  7.22407983,  10.87745653,  11.41121207,   7.90783438,
          7.9247746 ,   8.65868113],
       [ 17.38348778,  20.45298608,  26.79212272,  18.35081771,
         16.33254597,  21.03681988],
       [ 25.52436238,  30.81258569,  39.17026312,  29.74471464,
         30.21209202,  31.59657941],
       [ 41.37237874,  39.35382498,  49.52985279,  39.97085128,
         40.57167453,  42.02291898],
       [ 55.16848018,  53.06651488,  58.25459758,  52.79938788,
         50.46416238,  54.30095668],
       [ 66.86263398,  64.19348203,  68.56416488,  65.86147957,
         61.50771588,  65.72819909],
       [ 78.84040664,  77.23888912,  79.50761033,  77.97271456,
         73.86915511,  77.78936354],
       [ 88.09896352,  88.26577468,  88.71614473,  88.59921463,
         85.81356598,  90.08410798],
       [ 97.99146022, 103.16289803,  98.40843628,  97.92450087,
         94.62171448, 101.59474198],
       [108.18424183, 112.42147788, 109.73559382, 109.15156888,
        108.03413022, 114.4